<h1>Averegaing Aggregation ConchV1</h1>
<p><b>Methodology</b><br>
<p>We have a slide broken into N patches, and each patch has 5 crop views, each giving a 512-d vector.<br>
First, the 5 crop views per patch are averaged into one 512-d vector — this gives you one clean representation per patch.(i.e.) 5x512 -> 1x512<br>
Then, all patch vectors across the slide are averaged into a single 512-d vector — this becomes our slide-level feature.(i.e.) Nx512 -> 1x512 where N = No of patches<br>

In [ ]:
import os
import torch

def average_patch_features_ConchV1(source_root, dest_root):
    """
    Aggregation: Simple Average Pooling
    
    Per .pt file  : (5, 512) → mean over 5 crops → (1, 512)
    Per subfolder : mean over all patches          → (1, 512)  [slide-level]
    """
    os.makedirs(dest_root, exist_ok=True)

    for subfolder_name in os.listdir(source_root):
        subfolder_path = os.path.join(source_root, subfolder_name)
        if not os.path.isdir(subfolder_path):
            continue

        pt_files = [f for f in os.listdir(subfolder_path) if f.endswith('.pt')]
        if not pt_files:
            print(f"⚠ No .pt files in {subfolder_name}, skipping...")
            continue

        all_features = []

        for pt_file in pt_files:
            file_path = os.path.join(subfolder_path, pt_file)
            try:
                data = torch.load(file_path, map_location="cpu", weights_only=True)

                # ── ConchV1 specific shape check: (5, 512) ─────────────────
                if data.ndim != 2 or data.shape != torch.Size([5, 512]):
                    raise ValueError(f"Unexpected shape {data.shape}, expected (5, 512)")

                # Average over 5 crops → (1, 512)
                mean_feature = data.mean(dim=0, keepdim=True)
                all_features.append(mean_feature)

            except Exception as e:
                print(f"❌ Error loading {file_path}: {e}")
                continue

        if not all_features:
            print(f"⚠ No valid .pt files processed in {subfolder_name}")
            continue

        # Stack all patch features → (N, 512) then average → (1, 512)
        final_feature = torch.cat(all_features, dim=0).mean(dim=0, keepdim=True)

        dest_file_path = os.path.join(dest_root, f"{subfolder_name}.pt")
        torch.save(final_feature, dest_file_path)
        print(f"✅ [{subfolder_name}] {len(pt_files)} patches → aggregated to {tuple(final_feature.shape)}")

# ─── RUN ──────────────────────────────────────────────────────────────────────
source_path = r"D:\Aamir Gulzar\KSA_project2\dataspiapet\Features\Conch_v1_fivecrop"
dest_path   = r"D:\Aamir Gulzar\KSA_project2\dataset\slide_aggregation\Averaging\ConchV1"
average_patch_features_ConchV1(source_path, dest_path)
# ```

# ---

# ## What Changed from CONCH vs Virchow2

# | | CONCH | Virchow2 |
# |---|---|---|
# | Shape check | `(5, D)` generic | `(5, 2560)` explicit |
# | `torch.load` | no flags | `map_location="cpu", weights_only=True` |
# | Print on save | path only | path + patch count + output shape |

# ---

# ## The Aggregation Flow
# ```
# Subfolder (= one slide)
# │
# ├── patch_001.pt  (5, 2560) → mean over crops → (1, 2560)
# ├── patch_002.pt  (5, 2560) → mean over crops → (1, 2560)
# ├── patch_003.pt  (5, 2560) → mean over crops → (1, 2560)
# │         ...
# └── patch_N.pt    (5, 2560) → mean over crops → (1, 2560)
#          │
#          ▼ stack → (N, 2560)
#          ▼ mean  → (1, 2560)
#          │
#          └── saved as subfolder_name.pt  ← slide-level feature

✅ [TCGA-3L-AA1B_nonMSIH] 1878 patches → aggregated to (1, 512)
✅ [TCGA-4N-A93T_nonMSIH] 4240 patches → aggregated to (1, 512)
✅ [TCGA-5M-AAT4_nonMSIH] 6411 patches → aggregated to (1, 512)
✅ [TCGA-5M-AAT6_MSIH] 4320 patches → aggregated to (1, 512)
✅ [TCGA-5M-AATE_nonMSIH] 7583 patches → aggregated to (1, 512)
✅ [TCGA-A6-2671_nonMSIH] 710 patches → aggregated to (1, 512)
✅ [TCGA-A6-2681_nonMSIH] 840 patches → aggregated to (1, 512)
✅ [TCGA-A6-2685_nonMSIH] 2428 patches → aggregated to (1, 512)
✅ [TCGA-A6-2686_MSIH] 2258 patches → aggregated to (1, 512)
✅ [TCGA-A6-3807_nonMSIH] 2337 patches → aggregated to (1, 512)
✅ [TCGA-A6-4105_nonMSIH] 2365 patches → aggregated to (1, 512)
✅ [TCGA-A6-4107_nonMSIH] 836 patches → aggregated to (1, 512)
✅ [TCGA-A6-5657_nonMSIH] 1047 patches → aggregated to (1, 512)
✅ [TCGA-A6-5660_nonMSIH] 1016 patches → aggregated to (1, 512)
✅ [TCGA-A6-5661_MSIH] 875 patches → aggregated to (1, 512)
✅ [TCGA-A6-5662_nonMSIH] 532 patches → aggregated to (1, 512)
✅ [TCG

<h1>PAIP Avergaing</h1>

In [3]:
import os
import torch

def average_patch_features_virchow2(source_root, dest_root):
    """
    Aggregation: Simple Average Pooling
    
    Per .pt file  : (5, 2560) → mean over 5 crops → (1, 2560)
    Per subfolder : mean over all patches          → (1, 2560)  [slide-level]
    """
    os.makedirs(dest_root, exist_ok=True)

    for subfolder_name in os.listdir(source_root):
        subfolder_path = os.path.join(source_root, subfolder_name)
        if not os.path.isdir(subfolder_path):
            continue

        pt_files = [f for f in os.listdir(subfolder_path) if f.endswith('.pt')]
        if not pt_files:
            print(f"⚠ No .pt files in {subfolder_name}, skipping...")
            continue

        all_features = []

        for pt_file in pt_files:
            file_path = os.path.join(subfolder_path, pt_file)
            try:
                data = torch.load(file_path, map_location="cpu", weights_only=True)

                # ── ConchV1 specific shape check: (5, 512) ─────────────────
                if data.ndim != 2 or data.shape != torch.Size([5, 512]):
                    raise ValueError(f"Unexpected shape {data.shape}, expected (5, 512)")

                # Average over 5 crops → (1, 512)
                mean_feature = data.mean(dim=0, keepdim=True)
                all_features.append(mean_feature)

            except Exception as e:
                print(f"❌ Error loading {file_path}: {e}")
                continue

        if not all_features:
            print(f"⚠ No valid .pt files processed in {subfolder_name}")
            continue

        # Stack all patch features → (N, 512) then average → (1, 512)
        final_feature = torch.cat(all_features, dim=0).mean(dim=0, keepdim=True)

        dest_file_path = os.path.join(dest_root, f"{subfolder_name}.pt")
        torch.save(final_feature, dest_file_path)
        print(f"✅ [{subfolder_name}] {len(pt_files)} patches → aggregated to {tuple(final_feature.shape)}")

# ─── RUN ──────────────────────────────────────────────────────────────────────
source_path = r"D:\Aamir Gulzar\KSA_project2\paip_data\Features\Conch_v1_fivecrop_Updated"
dest_path   = r"D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Averaging\ConchV1"
average_patch_features_ConchV1(source_path, dest_path)
# ```

# ---

# ## What Changed from CONCH vs Virchow2

# | | CONCH | Virchow2 |
# |---|---|---|
# | Shape check | `(5, D)` generic | `(5, 2560)` explicit |
# | `torch.load` | no flags | `map_location="cpu", weights_only=True` |
# | Print on save | path only | path + patch count + output shape |

# ---

# ## The Aggregation Flow
# ```
# Subfolder (= one slide)
# │
# ├── patch_001.pt  (5, 2560) → mean over crops → (1, 2560)
# ├── patch_002.pt  (5, 2560) → mean over crops → (1, 2560)
# ├── patch_003.pt  (5, 2560) → mean over crops → (1, 2560)
# │         ...
# └── patch_N.pt    (5, 2560) → mean over crops → (1, 2560)
#          │
#          ▼ stack → (N, 2560)
#          ▼ mean  → (1, 2560)
#          │
#          └── saved as subfolder_name.pt  ← slide-level feature

✅ [training_data_01_MSIH] 3196 patches → aggregated to (1, 512)
✅ [training_data_02_nonMSIH] 4412 patches → aggregated to (1, 512)
✅ [training_data_03_nonMSIH] 1955 patches → aggregated to (1, 512)
✅ [training_data_04_nonMSIH] 3198 patches → aggregated to (1, 512)
✅ [training_data_05_MSIH] 4073 patches → aggregated to (1, 512)
✅ [training_data_06_MSIH] 3776 patches → aggregated to (1, 512)
✅ [training_data_07_nonMSIH] 2740 patches → aggregated to (1, 512)
✅ [training_data_08_nonMSIH] 51 patches → aggregated to (1, 512)
✅ [training_data_09_nonMSIH] 2248 patches → aggregated to (1, 512)
✅ [training_data_10_nonMSIH] 2452 patches → aggregated to (1, 512)
✅ [training_data_11_nonMSIH] 2893 patches → aggregated to (1, 512)
✅ [training_data_12_MSIH] 4691 patches → aggregated to (1, 512)
✅ [training_data_13_nonMSIH] 5530 patches → aggregated to (1, 512)
✅ [training_data_14_nonMSIH] 2674 patches → aggregated to (1, 512)
✅ [training_data_15_nonMSIH] 3834 patches → aggregated to (1, 512)
✅ [traini